# TensorDataset verification

These checks verify sample counting, indexing, input validation, empty tensors, and the single-tensor case. Each code cell uses assertions so a regression fails visibly.

In [3]:
from pathlib import Path
import sys
import numpy as np

# Make the notebook runnable from either the repository root or this folder.
repo_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "tinytorch").is_dir())
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from tinytorch.data_loader import TensorDataset
from tinytorch.tensor import Tensor_CP

In [4]:
def assert_raises(exception_type, operation):
    try:
        operation()
    except exception_type:
        return
    except Exception as error:
        raise AssertionError(
            f"Expected {exception_type.__name__}, got {type(error).__name__}"
        ) from error
    raise AssertionError(f"Expected {exception_type.__name__}, but nothing was raised")

## Aligned features and labels

In [5]:
features = Tensor_CP([[1, 2], [3, 4], [5, 6], [7, 8]])
labels = Tensor_CP([0, 1, 1, 0])
dataset = TensorDataset(features, labels)

assert len(dataset) == 4  # samples, not number of stored tensors
first_sample = dataset[0]
assert isinstance(first_sample, tuple)
assert len(first_sample) == 2
assert all(isinstance(value, Tensor_CP) for value in first_sample)
assert np.array_equal(first_sample[0].data, np.array([1, 2], dtype=np.float32))
assert first_sample[1].data.item() == 0
print("PASS: four aligned samples have length 4 and return two Tensor_CP objects")

PASS: four aligned samples have length 4 and return two Tensor_CP objects


## Invalid inputs

In [6]:
assert_raises(
    ValueError,
    lambda: TensorDataset(Tensor_CP([[1], [2], [3], [4]]), Tensor_CP([0, 1, 2])),
)
assert_raises(TypeError, lambda: TensorDataset([1, 2, 3, 4]))
assert_raises(TypeError, lambda: TensorDataset(np.array([1, 2, 3, 4])))
assert_raises(ValueError, lambda: TensorDataset(Tensor_CP(42)))
print("PASS: mismatched lengths, list/array inputs, and scalar tensors are rejected")

PASS: mismatched lengths, list/array inputs, and scalar tensors are rejected


## Empty and single-tensor datasets

In [7]:
empty_features = Tensor_CP(np.empty((0, 3), dtype=np.float32))
empty_labels = Tensor_CP(np.empty((0,), dtype=np.float32))
empty_dataset = TensorDataset(empty_features, empty_labels)
assert len(empty_dataset) == 0

single_dataset = TensorDataset(Tensor_CP([10, 20, 30, 40]))
assert len(single_dataset) == 4
single_sample = single_dataset[0]
assert isinstance(single_sample, tuple) and len(single_sample) == 1
assert isinstance(single_sample[0], Tensor_CP)
assert single_sample[0].data.item() == 10
print("PASS: aligned empty tensors and a single tensor both work")
print("All requested TensorDataset checks passed.")

PASS: aligned empty tensors and a single tensor both work
All requested TensorDataset checks passed.
